# 03 - Let the worker act for a user

**Goal:** show the same child agent identity acting **on behalf of** a real signed-in user (OBO). The agent keeps its badge; it also gets permission to act for that user.

**Before starting:** complete [00 Identity](./00_Identity_Setup.ipynb) and [01 Permissions](./01_Agent_Permissions.ipynb). Reuse their blueprint, child, MCP API, and settings. A model, Application Insights, and Agent 365 licensing are NOT required for the permission checks here.

**Two separate permission links:** browser -> blueprint (`access_as_user`); child acting for the user -> MCP (`Workshop.Read`). Approving one does not approve the other.

Steps 1-3 are one-time administrator setup. Start a fresh kernel for steps 4-5.

For the school-assistant analogy, token diagrams, and scope-versus-role comparison, read [the workshop guide](./WORKSHOP_START_HERE.md). Signing in does not give the agent every permission the user has; an API handling business records must also enforce that user's access to those records.

## 1. Register the sign-in window - once

**An authorized administrator does this.** The child and blueprint are not interactive browser apps, so use a separate ordinary client.

1. Entra > **App registrations > New registration**: `agent365-workshop-browser`, single tenant.
2. **Authentication > Add a platform > Mobile and desktop applications**: add `http://localhost:8400`. Enable public client flows if the tenant UI requires it. Do not create a secret.
3. Save the browser **client ID** to `MCP_WORKSHOP_BROWSER_CLIENT_ID` in the private `.env`.

**Why:** this app opens the user sign-in window. It is not the agent's identity and does not need direct MCP or Graph permissions.

## 2. Let the browser call the blueprint - once

**The authorized blueprint administrator does this.** Open **Entra ID > Agents > Agent blueprints > your blueprint > Developer settings > Manifest** (or **Expose an API**, if available).

1. Include `api://<BLUEPRINT-CLIENT-ID>` in `identifierUris`.
2. Set `api.requestedAccessTokenVersion` to `2`.
3. Add an enabled delegated scope `access_as_user` to `api.oauth2PermissionScopes`, with a new unique scope ID and consent text describing access to this workshop agent.
4. In `api.preAuthorizedApplications`, add the browser: `appId` = browser client ID; `delegatedPermissionIds` = array containing that scope ID. Preserve existing entries and manifest settings.
5. In the **browser app > API permissions > Add a permission > My APIs**, choose the blueprint's delegated `access_as_user`. Have the administrator grant required consent.

**Why:** the user first gets a token **for the blueprint**, not for Graph or MCP. The next exchange trades it for a downstream MCP token. These are different passes for different doors.

[Microsoft's agent OBO flow](https://learn.microsoft.com/entra/agent-id/agent-on-behalf-of-oauth-flow)

## 3. Allow this child to use MCP for this user - once

**An authorized consent administrator does this.** Use a demo user in the same tenant. Notebook 01 must already have created the API's delegated `Workshop.Read` scope.

In PowerShell 7 with Microsoft.Graph.Applications and Microsoft.Graph.Identity.SignIns modules, replace all four placeholders. This grants only the named user, child, and API, not all users.

```powershell
$ErrorActionPreference = 'Stop'
Connect-MgGraph -TenantId '<TENANT-ID>' -Scopes 'Application.Read.All','DelegatedPermissionGrant.ReadWrite.All' -UseDeviceAuthentication -ContextScope Process -ErrorAction Stop
$childObjectId = '<CHILD-OBJECT-ID>'
$apiClientId = '<MCP-API-CLIENT-ID>'
$userObjectId = '<DEMO-USER-OBJECT-ID>'
$apis = @(Get-MgServicePrincipal -Filter "appId eq '$apiClientId'" -Property Id)
if ($apis.Count -ne 1) { throw 'Expected one MCP API principal.' }
$resourceId = $apis[0].Id
$grants = @(Get-MgOauth2PermissionGrant -Filter "clientId eq '$childObjectId'" -All | Where-Object { $_.ResourceId -eq $resourceId -and $_.ConsentType -eq 'Principal' -and $_.PrincipalId -eq $userObjectId })
if ($grants.Count -gt 1) { throw 'Multiple grants; review before changing consent.' }
if ($grants.Count -eq 1) {
    if ('Workshop.Read' -notin ($grants[0].Scope -split ' ')) {
        Update-MgOauth2PermissionGrant -OAuth2PermissionGrantId $grants[0].Id -Scope (($grants[0].Scope + ' Workshop.Read').Trim())
    }
} else {
    $body = @{clientId=$childObjectId; resourceId=$resourceId; consentType='Principal'; principalId=$userObjectId; scope='Workshop.Read'; expiryTime=[DateTime]::UtcNow.AddDays(7).ToString('yyyy-MM-ddTHH:mm:ssZ')}
    Invoke-MgGraphRequest -Method POST -Uri 'https://graph.microsoft.com/v1.0/oauth2PermissionGrants' -Body $body
}
```

**Why:** an application role for agent-only calls does not grant delegated access. Allow consent to propagate. Record and revoke workshop grants afterward; do not rely on a time field alone for cleanup.

## 4. Compare the same agent with and without a user

**Setup needed:** steps 1-3, notebook 01's app-role grant, project kernel, and private `.env` settings. Restart after settings changes. Start from the project root or `Agent_365`.

First run agent-only mode again. It should deny `user_context`. Then sign in as the approved demo user; the OBO call should return the verified user object ID and the **same child client ID**. Never paste tokens into cells.

In [ ]:
from pathlib import Path
import sys

root = Path.cwd().parent if Path.cwd().name == 'Agent_365' else Path.cwd()
sys.path.insert(0, str(root))
from Agent_365.mcp_workshop import MCPSettings, sign_in_agent, explore_mcp_identity

settings = MCPSettings.from_env()

In [ ]:
agent_access = await sign_in_agent(settings)
await explore_mcp_identity(settings, agent_access)

In [ ]:
user_access = await sign_in_agent(settings, on_behalf_of_user=True)
await explore_mcp_identity(settings, user_access)

**Read the evidence:** agent-only discovery lists `who_am_i`; OBO discovery also lists `user_context`. The server derives the user from the signed token, not from a name supplied by the model. The helper never prints the token.

**If it fails:** verify the localhost redirect and browser ID (step 1), the browser-to-blueprint consent (step 2), and the per-user child-to-MCP grant (step 3). A Graph user token cannot replace a blueprint user token.

**Where to look:** Entra sign-in logs for authentication, and these tool results for authorization. This notebook does not configure Azure or Agent 365 tracing.

## 5. Optional: let the agent explain the user's permission

**Extra setup required:** notebook 02 step 1's Azure model settings, CLI sign-in, and model inference access. This is a billable model call. It does not require Application Insights.

Only read-only identity results, including the user's object ID, go to the Azure model. No secret or bearer token goes to it. The model still uses your CLI identity; MCP uses the child's OBO token.

In [ ]:
from Agent_365.mcp_workshop import run_mcp_assistant

await run_mcp_assistant(settings, user_access)

**Say:** "The worker kept its own badge. The user gave it an extra, limited permission; it did not become the user or gain every user permission."

Clear identity outputs before sharing. Revoke temporary grants and remove unused credentials after the workshop.

Next: [04 - Agent 365 observability](./04_Agent365_Observability.ipynb).